### Käsitsi kureeritud elus/koht nimekirjade alusel sõnade jaotus

In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import powerlaw as pwl
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
from common_sql import update_table, create_count_table, create_left_join_table

In [ ]:
# transaktsioonide andmebaas
transaction_db = "../example_data/v33_subset.db"

# Siia salvestuvad loodavad tabelid
vp_data_db = "../example_data/vp_data_actors.db"


In [49]:
con = sqlite3.connect(vp_data_db)
cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{transaction_db}" AS trans')

In [ ]:
# define table names 

transactions = "transaction_v2"

# temp loendus tabelid
root_eluskoht_cnts_base = "transaction_v2_obl_root_eluskoht_counts_base"
root_eluskoht_cnts_elus = "transaction_v2_obl_root_eluskoht_counts_elus"
root_eluskoht_cnts_koht = "transaction_v2_obl_root_eluskoht_counts_koht"

# vahetabel ja lõplik tabel
counts1 = "transaction_v2_obl_root_eluskoht_counts_1"
counts = "transaction_v2_obl_root_eluskoht_counts"

## Create necessary table

In [28]:
# base table with root counts
create_count_table(con, transactions, root_eluskoht_cnts_base,
                  ["lemma"], "lemma", "lemmacnt", "deprel = 'obl'", ["lemma"])

# table counts elus 
create_count_table(con, transactions, root_eluskoht_cnts_elus,
                  ["lemma"], "lemma", "elus_cnt", "deprel = 'obl' and elus = 'YES'", ["lemma"])

# tbl count koht
create_count_table(con, transactions, root_eluskoht_cnts_koht,
                  ["lemma"], "lemma", "koht_cnt", "deprel = 'obl' and koht = 'YES'", ["lemma"])


In [31]:
# join everything into 1 table

create_left_join_table(con, source_tbl1=root_eluskoht_cnts_base, source_tbl2=root_eluskoht_cnts_elus,result_table=counts1,
                selected_columns=["tbl1.lemma", "tbl1.lemmacnt as lemma_cnt", "elus_cnt"],
                condition="tbl1.lemma=tbl2.lemma ")

create_left_join_table(con, source_tbl1=counts1, source_tbl2=root_eluskoht_cnts_koht,result_table=counts,
                selected_columns=["tbl1.lemma", "tbl1.lemma_cnt", "tbl1.elus_cnt", "koht_cnt"],
                condition="tbl1.lemma=tbl2.lemma ")

In [32]:
# update table null -> 0
update_table(con, counts, "elus_cnt", 0, "elus_cnt is null")
update_table(con, counts, "koht_cnt", 0, "koht_cnt is null")

In [47]:
query = """SELECT * from {tbl}
where elus_cnt!=koht_cnt
limit 20
""".format(tbl=counts)

source2 = pd.read_sql_query(query, con)
source2

,lemma,lemma_cnt,elus_cnt,koht_cnt
0,2toaline,1,0,1
1,3toaline,1,0,1
2,AAV,1,0,1
3,Aadress,1,0,1
4,Aafrika,1168,0,1168
5,Abikaasa,2,2,0
6,Abilinnapea,1,1,0
7,Afgaan,1,1,0
8,Agent,2,2,0
9,Ajakirjanik,2,2,0


In [51]:
query = """
SELECT * 
from 
{tbl}
where elus_cnt!=lemma_cnt and elus_cnt>0
limit 20
""".format(tbl=counts)

source2 = pd.read_sql_query(query, con)
source2

,lemma,lemma_cnt,elus_cnt,koht_cnt


In [52]:
con.close()